# Which stations are full and free at the same times?

Step 1: a matrix with one row per snapshot (about one per minute) and one column per station, holding the
**share of free docks** docks / (bikes + docks), then the correlation between every pair of columns.

## 1. Load

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = Path("data_1")
TZ = "Europe/Paris"
pd.set_option("display.width", 160)

snap = pd.read_csv(DATA / "snapshots.csv", usecols=["src_ts", "station_id", "bikes", "docks"])
names = pd.read_csv(DATA / "stations.csv", encoding="utf-8").set_index("station_id")["name"]
snap = snap[snap.bikes + snap.docks > 0]  # drop offline snapshots (0 bikes and 0 docks)
print(f"{len(snap):,} rows, {snap.station_id.nunique()} stations")

## 2. Correlation of the share of free docks

share = docks / (bikes + docks): 0 = full, 1 = empty. Dividing by bikes + docks puts every station on the same 0–1 scale,
whatever its size.

How to read the correlation r of two stations:
- **close to +1:** they are nearly full at the same times and free at the same times
- **close to 0:** one station's share says nothing about the other's
- **close to −1:** when one is nearly full, the other is nearly empty

In [ ]:
share = snap.assign(share=snap.docks / (snap.bikes + snap.docks)).pivot(index="src_ts", columns="station_id", values="share")
share = share.dropna(axis=1)                         # keep stations seen in every snapshot
share = share.loc[:, (share.diff().iloc[1:] != 0).sum() >= 30]  # and that change at least 30 times
print(f"matrix: {share.shape[0]} snapshots × {share.shape[1]} stations")

R = share.corr()  # correlation matrix, stations × stations
R = R.rename_axis(index="id A", columns="id B")
pairs = R.where(np.triu(np.ones(R.shape, bool), k=1)).stack().dropna().rename("r").reset_index()  # each pair once
pairs["station A"] = pairs["id A"].map(names)
pairs["station B"] = pairs["id B"].map(names)
pairs = pairs.sort_values("r")
print(f"{len(pairs):,} pairs, median r = {pairs.r.median():.2f}")

cols = ["r", "station A", "station B", "id A", "id B"]
print("\nhighest correlation:")
display(pairs.tail(10)[::-1][cols].round(3))
print("about zero correlation:")
display(pairs[pairs.r.abs() < 0.02].sample(10, random_state=0)[cols].round(3))
print("most negative correlation:")
display(pairs.head(10)[cols].round(3))

One example of each: the two stations' share of free docks over time.

In [ ]:
time = pd.to_datetime(share.index, unit="s", utc=True).tz_convert(TZ)
examples = {
    "highest r": pairs.iloc[-1],
    "r about 0": pairs[pairs.r.abs() < 0.02].sample(1, random_state=0).iloc[0],
    "most negative r": pairs.iloc[0],
}
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
for ax, (title, row) in zip(axes, examples.items()):
    for sid, color in ((row["id A"], "tab:blue"), (row["id B"], "tab:orange")):
        ax.plot(time, share[sid], color=color, lw=1.2, label=names.get(sid, sid))
    ax.set(ylim=(-0.03, 1.03), ylabel="free docks / slots", title=f"{title}: r = {row.r:.2f}")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", frameon=False)
axes[-1].set_xlabel("time (Paris)")
plt.tight_layout()
plt.show()

## 3. Hierarchical clustering

Distance between two stations: d = 1 − r (0 = identical curves, 1 = unrelated, 2 = mirror images). Start with every
station alone, then repeatedly merge the two closest groups, where the distance between two groups is the **average**
distance between their stations (average linkage). The merges form a tree (dendrogram).

Cut the tree at height 1 − `MIN_R`: two groups are only merged if their stations correlate on average by at least
`MIN_R`. The cut decides the number of clusters and their sizes, instead of choosing them by hand. Groups smaller than
`MIN_SIZE` stations are left in no cluster.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import squareform

MIN_R = 0.7     # cut: groups whose stations correlate on average less than this are not merged
MIN_SIZE = 10   # smaller groups are left in no cluster
N_SHOW = 8      # clusters drawn (the largest)

D = (1 - R.to_numpy()).clip(min=0)  # clip: rounding can give -1e-16 on the diagonal
np.fill_diagonal(D, 0)
tree = linkage(squareform(D, checks=False), method="average")

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={"width_ratios": [2, 1, 1]})
dendrogram(tree, truncate_mode="lastp", p=40, no_labels=True, color_threshold=0, above_threshold_color="tab:blue", ax=ax1)
ax1.set(ylabel="distance 1 − r", title="Top of the tree (last 40 merges)")
# how the choice of MIN_R changes the result
grid = np.round(np.arange(0.4, 0.96, 0.05), 2)
n_clusters, n_stations = [], []
for r_cut in grid:
    s = pd.Series(fcluster(tree, t=1 - r_cut, criterion="distance")).value_counts()
    n_clusters.append((s >= MIN_SIZE).sum())
    n_stations.append(s[s >= MIN_SIZE].sum())
for ax, y, title in ((ax2, n_clusters, f"clusters with ≥ {MIN_SIZE} stations"), (ax3, n_stations, "stations in those clusters")):
    ax.plot(grid, y, marker="o", color="tab:blue")
    ax.axvline(MIN_R, color="0.3", ls="--", lw=1)  # current choice
    ax.set(xlabel="MIN_R", title=title)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

label = pd.Series(fcluster(tree, t=1 - MIN_R, criterion="distance"), index=R.index.rename("station_id"))
sizes = label.value_counts()
kept = sizes[sizes >= MIN_SIZE].index  # largest first
hclusters = {f"H{i + 1}": list(label.index[label == c]) for i, c in enumerate(kept)}
print(f"{len(sizes)} groups at the cut; {len(hclusters)} with at least {MIN_SIZE} stations, holding "
      f"{sum(map(len, hclusters.values()))} of {len(label)} stations")

rows = []
for name, members in hclusters.items():
    within = R.loc[members, members].to_numpy()[np.triu_indices(len(members), 1)].mean()
    rows.append({"cluster": name, "stations": len(members), "mean r within": round(within, 2),
                 "first stations": ", ".join(names.get(s, str(s)) for s in members[:4])})
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(rows).set_index("cluster"))

# a row of NaN just before the first snapshot after a data gap, so the lines break instead of crossing the gap
after_gap = pd.Series(time).diff().gt(pd.Timedelta("5min")).to_numpy()
breaks = pd.DataFrame(np.nan, index=time[after_gap] - pd.Timedelta("1s"), columns=share.columns)
by_time = pd.concat([share.set_axis(time), breaks]).sort_index()

shown = list(hclusters)[:N_SHOW]
fig, axes = plt.subplots(len(shown), 1, figsize=(13, 2.6 * len(shown)), sharex=True)
for ax, name in zip(np.atleast_1d(axes), shown):
    members = hclusters[name]
    ax.plot(by_time.index, by_time[members].to_numpy(), color="tab:blue", lw=0.5, alpha=0.3)
    ax.plot(by_time.index, by_time[members].median(axis=1), color="tab:blue", lw=2.5)
    ax.set(ylim=(0, 1), ylabel="free docks / slots", title=f"{name}: {len(members)} stations, median in bold")
    ax.grid(alpha=0.3)
ax.set_xlabel("time (Paris)")
plt.tight_layout()
plt.show()

The original data: number of free docks of every station, one panel per cluster.

In [ ]:
docks = snap.pivot(index="src_ts", columns="station_id", values="docks").loc[share.index, share.columns]
docks_by_time = pd.concat([docks.set_axis(time), breaks]).sort_index()  # same NaN rows at the data gap as by_time

fig, axes = plt.subplots(len(hclusters), 1, figsize=(13, 2.6 * len(hclusters)), sharex=True)
for ax, (name, members) in zip(np.atleast_1d(axes), hclusters.items()):
    ax.plot(docks_by_time.index, docks_by_time[members].to_numpy(), color="tab:blue", lw=0.6, alpha=0.3)
    ax.set(ylim=(0, None), ylabel="free docks", title=f"{name}: {len(members)} stations")
    ax.grid(alpha=0.3)
ax.set_xlabel("time (Paris)")
plt.tight_layout()
plt.show()

## 4. New: correlate the changes, not the levels

The correlation in section 2 compares the **levels** of the share of free docks. Two smooth curves that each go up and down
once a day correlate strongly by chance, even when their timing is unrelated. With only two days of data they also have very
few independent ups and downs. This section tests a second measure that asks literally "do they rise and fall together?":

- average the share over `STEP` (30 min), then take the **change from one step to the next**; r is the correlation of
  those changes
- **every station is kept**: an offline snapshot (0 bikes and 0 docks) is NaN instead of removing the whole station, and each
  pair is computed on the steps where both stations have data (`min_periods`). Only stations that almost never move are
  dropped, because a constant series has no correlation
- a step with no data (the gap on 28 Oct) is NaN, so no change is measured across it

Shorter steps give more data points but mostly noise from single trips (median r within 500 m: 0.02 at 5 min, 0.07 at 15
min, 0.12 at 30 min, 0.23 at 60 min, with only 49 steps at 60 min). 30 min is a compromise; revisit it with more days.

In [ ]:
STEP = "30min"   # changes are measured over this step
MIN_STEPS = 30   # a pair needs at least this many steps where both stations have data

# every station, every snapshot: offline snapshots (dropped from snap above) become NaN instead of removing the station
wide = snap.assign(share=snap.docks / (snap.bikes + snap.docks)).pivot(index="src_ts", columns="station_id", values="share")
wide.index = pd.to_datetime(wide.index, unit="s", utc=True).tz_convert(TZ)
# mean share per step, then the change from one step to the next; a step with no snapshot (data gap) is NaN,
# so the changes on either side of it are NaN too instead of spanning the gap
change = wide.resample(STEP).mean().diff()
change = change.loc[:, (change.abs() > 0).sum() >= MIN_STEPS]  # a station that (almost) never moves has no correlation
Rc = change.corr(min_periods=MIN_STEPS).rename_axis(index="id A", columns="id B")  # pairwise: each pair uses the steps where both have data
pairs_c = Rc.where(np.triu(np.ones(Rc.shape, bool), k=1)).stack().dropna().rename("r").reset_index()
pairs_c["station A"] = pairs_c["id A"].map(names)
pairs_c["station B"] = pairs_c["id B"].map(names)
pairs_c = pairs_c.sort_values("r")
print(f"matrix: {change.shape[0]} steps of {STEP} × {change.shape[1]} stations "
      f"({len(Rc.index.difference(R.index))} of them not in the level matrix above)")
print(f"{len(pairs_c):,} pairs, median r = {pairs_c.r.median():.2f}")
cols = ["r", "station A", "station B", "id A", "id B"]
print("\nhighest correlation of changes:")
display(pairs_c.tail(10)[::-1][cols].round(3))
print("most negative correlation of changes:")
display(pairs_c.head(10)[cols].round(3))


### How do the two methods compare?

- **Left:** the same pairs scored by both methods.
- **Middle:** median r against the distance between the two stations. Real co-movement (same neighbourhood, same commuters)
  should fade with distance.
- **Right:** r of the real pairs compared with **chance**. Chance comes from random pairs after rotating one series by a random
  offset (at least a quarter of the run). That keeps each curve's own shape and smoothness but breaks any shared timing. The
  cell prints the r that 99% of these shifted pairs stay below. A real pair needs an r above that before it means anything.

In [ ]:
coords = pd.read_csv(DATA / "stations.csv", encoding="utf-8").set_index("station_id")[["lat", "lon"]]


def with_distance(p):
    # distance in km between the two stations of each pair (flat-earth approximation, fine at city scale)
    a, b = coords.reindex(p["id A"]).to_numpy(), coords.reindex(p["id B"]).to_numpy()
    dy = (b[:, 0] - a[:, 0]) * 111.2
    dx = (b[:, 1] - a[:, 1]) * 111.2 * np.cos(np.radians(a[:, 0]))
    return p.assign(km=np.hypot(dx, dy))


def shifted_r(M, n_pairs=3000, seed=0):
    # chance level: correlate random pairs after rotating one series by a random offset (at least a quarter of the
    # run), which keeps each series' own smoothness and daily shape but breaks any timing they share
    rng = np.random.default_rng(seed)
    X = M.to_numpy()
    out = []
    for _ in range(n_pairs):
        i, j = rng.choice(X.shape[1], 2, replace=False)
        lag = rng.integers(len(X) // 4, len(X) - len(X) // 4)
        out.append(pd.Series(X[:, i]).corr(pd.Series(np.roll(X[:, j], lag)), min_periods=MIN_STEPS))
    return np.array(out)


both = with_distance(pairs.merge(pairs_c, on=["id A", "id B"], suffixes=("_level", "_change")))
levels = {"level (existing)": (pairs, share, "tab:blue"), f"change per {STEP} (new)": (pairs_c, change, "tab:orange")}
bins = [0, 0.5, 1, 2, 4, 8, 16, 40]

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].hexbin(both.r_level, both.r_change, gridsize=60, bins="log", cmap="Blues")
ax[0].set(xlabel="r of levels (existing)", ylabel=f"r of changes per {STEP} (new)",
          title=f"same pairs, both methods (correlation {both.r_level.corr(both.r_change):.2f})")
for name, (p, M, color) in levels.items():
    p = with_distance(p)
    by_km = p.groupby(pd.cut(p.km, bins), observed=True).r.median()
    ax[1].plot([str(b) for b in by_km.index], by_km.to_numpy(), marker="o", color=color, label=name)
    null = shifted_r(M)
    ax[2].hist(p.r, bins=100, range=(-1, 1), density=True, histtype="step", color=color, lw=1.5, label=f"{name}: real pairs")
    ax[2].hist(null, bins=100, range=(-1, 1), density=True, histtype="step", color=color, ls=":", lw=1.5,
               label=f"{name}: chance (shifted)")
    hi = np.nanquantile(null, 0.99)
    print(f"{name}: 99% of shifted pairs have r < {hi:.2f}; {(p.r > hi).mean():.1%} of real pairs are above that")
ax[1].set(xlabel="distance between the two stations (km)", ylabel="median r", title="Does co-movement fade with distance?")
ax[1].tick_params(axis="x", rotation=30)
ax[2].set(xlabel="r", ylabel="density", title="Real pairs vs chance")
for a in ax[1:]:
    a.grid(alpha=0.3)
    a.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()


### Clusters of the changes, and where they are

Same hierarchical clustering as section 3, on d = 1 − r of the changes. The cut `MIN_R_C` is lower than `MIN_R` because
changes correlate less than levels, but it should stay above the chance threshold printed in the cell above. Maps: one
colour per cluster (the `N_SHOW` largest), grey = in no cluster. Left = section 3, right = this section.

In [ ]:
MIN_R_C = 0.35   # changes correlate less than levels, so the cut is lower than MIN_R

Dc = (1 - Rc.fillna(0).to_numpy()).clip(min=0)  # a pair with too few common steps counts as unrelated (r = 0)
np.fill_diagonal(Dc, 0)
tree_c = linkage(squareform(Dc, checks=False), method="average")
label_c = pd.Series(fcluster(tree_c, t=1 - MIN_R_C, criterion="distance"), index=Rc.index.rename("station_id"))
sizes_c = label_c.value_counts()
cclusters = {f"C{i + 1}": list(label_c.index[label_c == c]) for i, c in enumerate(sizes_c[sizes_c >= MIN_SIZE].index)}
print(f"{len(cclusters)} clusters with at least {MIN_SIZE} stations, holding {sum(map(len, cclusters.values()))} "
      f"of {len(label_c)} stations (existing method: {len(hclusters)} clusters, {sum(map(len, hclusters.values()))} stations)")

rows = []
for name, members in cclusters.items():
    within = Rc.loc[members, members].to_numpy()[np.triu_indices(len(members), 1)].mean()
    rows.append({"cluster": name, "stations": len(members), "mean r within": round(within, 2),
                 "first stations": ", ".join(names.get(s, str(s)) for s in members[:4])})
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(rows).set_index("cluster"))

# where the clusters are: one colour per cluster (the N_SHOW largest), grey = in no cluster
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=True, sharey=True)
for ax, (title, clusters, members_all) in zip(axes, [("levels (existing)", hclusters, R.index),
                                                     (f"changes per {STEP} (new)", cclusters, Rc.index)]):
    xy = coords.reindex(members_all).dropna()
    ax.scatter(xy.lon, xy.lat, s=4, color="0.85")
    for i, (name, members) in enumerate(list(clusters.items())[:N_SHOW]):
        xy = coords.reindex(members).dropna()
        ax.scatter(xy.lon, xy.lat, s=12, color=plt.cm.tab10(i), label=f"{name} ({len(members)})")
    ax.set(title=f"clusters of {title}", xlabel="longitude")
    ax.set_aspect(1 / np.cos(np.radians(48.86)))  # so distances look the same in both directions
    ax.legend(frameon=False, fontsize=8, loc="lower left")
axes[0].set_ylabel("latitude")
plt.tight_layout()
plt.show()


Each cluster over time: thin lines are the stations' changes per step, bold is the median. Above 0 the stations gain free
docks (bikes leave), below 0 they lose them (bikes arrive).

In [ ]:
# the clusters over time: the change in share per step, when each group fills (> 0: more free docks) and empties
fig, axes = plt.subplots(min(N_SHOW, len(cclusters)), 1, figsize=(13, 2.6 * min(N_SHOW, len(cclusters))), sharex=True)
for ax, name in zip(np.atleast_1d(axes), list(cclusters)[:N_SHOW]):
    members = cclusters[name]
    steps_c = change[members].asfreq(STEP)  # data-gap steps stay NaN, so the lines break there
    ax.plot(steps_c.index, steps_c.to_numpy(), color="tab:orange", lw=0.5, alpha=0.3)
    ax.plot(steps_c.index, steps_c.median(axis=1), color="tab:orange", lw=2.5)
    ax.axhline(0, color="0.3", lw=0.8)
    ax.set(ylim=(-0.5, 0.5), ylabel=f"change in share\nper {STEP}", title=f"{name}: {len(members)} stations, median in bold")
    ax.grid(alpha=0.3)
ax.set_xlabel("time (Paris)")
plt.tight_layout()
plt.show()
